In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from mlflow.models.signature import infer_signature
import mlflow
import mlflow.spark
import os
import time
import pandas as pd


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

mlflow.set_experiment("/Users/ar761179@gmail.com/taxi_fare_prediction")



In [0]:
df = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/silver_2019"
)

print("Total Clean Rows:", df.count())


In [0]:
feature_cols = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data_ml = assembler.transform(df) \
                   .select("features",
                           col("fare_amount").alias("label"),
                           "pickup_month")

In [0]:
train = data_ml.filter(col("pickup_month") <= 9)
test  = data_ml.filter(col("pickup_month") > 9)

print("Train Rows:", train.count())
print("Test Rows :", test.count())

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 100)

# Use 30% sample for training models (fast + stable)
train_sample = train.sample(fraction=0.3, seed=42)

In [0]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

results = {}


In [0]:
# ==========================================================
# 8️⃣ MODEL 1 – Linear Regression
# ==========================================================

with mlflow.start_run(run_name="LinearRegression"):

    start = time.time()

    lr = LinearRegression(maxIter=50)
    lr_model = lr.fit(train_sample)

    predictions = lr_model.transform(test)

    rmse = evaluator.evaluate(predictions)

    runtime = time.time() - start

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("training_time_sec", runtime)

    # Signature logging
    input_example = train_sample.limit(5).toPandas()
    example_pred = lr_model.transform(train_sample.limit(5)) \
                           .select("prediction") \
                           .toPandas()

    signature = infer_signature(input_example, example_pred)

    mlflow.spark.log_model(
    lr_model,
    artifact_path="model",
    dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
)

    results["LinearRegression"] = rmse

print("Linear Regression Done")

In [0]:
with mlflow.start_run(run_name="DecisionTree"):

    start = time.time()

    dt = DecisionTreeRegressor(
        maxDepth=6,
        maxBins=32
    )

    dt_model = dt.fit(train_sample)
    predictions = dt_model.transform(test)

    rmse = evaluator.evaluate(predictions)
    runtime = time.time() - start

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("training_time_sec", runtime)

    mlflow.spark.log_model(
        dt_model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    results["DecisionTree"] = rmse

print("Decision Tree Done")

In [0]:
with mlflow.start_run(run_name="RandomForest"):

    start = time.time()

    rf = RandomForestRegressor(
        numTrees=20,
        maxDepth=6,
        maxBins=32
    )

    rf_model = rf.fit(train_sample)
    predictions = rf_model.transform(test)

    rmse = evaluator.evaluate(predictions)
    runtime = time.time() - start

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("training_time_sec", runtime)

    mlflow.spark.log_model(
        rf_model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    results["RandomForest"] = rmse

print("Random Forest Done")



In [0]:
with mlflow.start_run(run_name="GBT"):

    start = time.time()

    gbt_train = train.sample(0.05, seed=42)

    gbt = GBTRegressor(
        maxIter=10,
        maxDepth=3,
        maxBins=32
    )

    gbt_model = gbt.fit(gbt_train)
    predictions = gbt_model.transform(test)
    rmse = evaluator.evaluate(predictions)
    runtime = time.time() - start

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("training_time_sec", runtime)

    mlflow.spark.log_model(
        gbt_model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
    )

    results["GBT"] = rmse

print("GBT Done")

In [0]:
print("\n================ FINAL RESULTS ================\n")
for model_name, rmse in results.items():
    print(model_name, "RMSE:", rmse)

In [0]:
print("Train total:", train.count())
print("GBT train sample:", gbt_train.count())